<img src="https://cads.ai/image/CADSLogo.png" style="height: 100px;" align=left>
<div align=right><svg height=100px viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg"><g clip-path="url(#clip1_4783_495)"><path d="M6.64803 14.103C7.89438 12.8566 8.595 11.1643 8.595 9.40177C8.595 7.63928 7.89378 5.94697 6.64803 4.70058L1.94697 0C0.701225 1.24639 0 2.9387 0 4.70119C0 6.46368 0.701225 8.15599 1.94697 9.40238L6.64742 14.103H6.64803Z" fill="#7B9AB5"></path><path d="M16.4845 14.5379C15.2388 13.2921 13.5459 12.5908 11.7841 12.5908C10.0222 12.5908 8.32936 13.2921 7.08301 14.5379L11.7841 19.239C13.0298 20.4848 14.7227 21.1861 16.4851 21.1861C18.2476 21.1861 19.9398 20.4848 21.1862 19.239L16.4851 14.5379H16.4845Z" fill="#7B9AB5"></path><path d="M1.95832 19.228C3.20468 20.4738 4.89693 21.1751 6.65938 21.1751V14.5269H0.0107422C0.0113472 16.2893 0.711968 17.9817 1.95832 19.228Z" fill="#7B9AB5"></path><path d="M18.2997 7.58717C17.0533 6.34138 15.3611 5.63953 13.598 5.64014C11.8356 5.64014 10.1433 6.34138 8.89697 7.58777L13.598 12.289L18.2997 7.58717Z" fill="#7B9AB5"></path></g></svg></div>

<a box-sizing="border-box" color-scheme=dark font-family="IBM Plex Sans" line-height=30px font-size=72px font-weight=700 align=center>© 2026 — Mehdi Lotfinejad ⨯ Azhar bin Ab Halim @ DATATWEETS</a>

# Routes and relay agents
Day 2 | Notebook 3 of 4

![A rail-style map: a START station, a yellow ROUTER junction, a dashed detour to a TOOLS station that returns to the router, and a solid track to the END station.](media/d2-03-hero-route-map.svg)

**You can re-run any practice until it works. Nothing here is recorded or marked.**

**Today in one sentence:** you route work through a graph, decide how much freedom the assistant gets, and meet a supervisor.

## Today's story

The Seri Maju Policy Assistant (fictional; Kuala Lumpur, Penang, Johor Bahru) now has tools and a memory. Next question: *who decides what happens next?* You'll build a small **graph** where a chatbot node and a tools node hand work back and forth, and you read the route a question takes. You'll decide **how much freedom** the assistant gets (the autonomy dial and its fence of guards) and meet a **supervisor** that hands work to two specialist desks.

**Fictional example, not legal or tax advice.*

## What you will be able to do

11. Build a graph with a tool node and a conditional edge, read the route a question takes, and stop a loop with a limit.
12. Set an autonomy boundary with guards (allow-list, hop budget, ask a human) and explain how a supervisor differs from a relay.

---

**Prerequisites:** `d2-02-*.ipynb` (tools) helps. Needs `langgraph` (see `.requirements.txt`).

**Words you do not know?** See the [Glossary](glossary.md).

In [1]:
# Answer switch (change it, then choose Run > Restart Kernel and Run All Cells).
#     False = you type the blanks.
#     True = Run All reads the reference answers.
USE_SOLUTIONS = globals().get("USE_SOLUTIONS", False)

In [2]:
%pip install "langchain>=0.3,<0.4" "langchain-core>=0.3,<0.4" "langchain-groq>=0.3,<0.4" "langchain-text-splitters>=0.3,<0.4" "langgraph>=0.6,<0.7" "pydantic>=2,<3" "scikit-learn>=1.4" "python-dotenv"

In [3]:
# ---- CANONICAL SETUP ----
# ---- HELPER - Run it, then ignore it. ----


## You can re-run any practice until it works. Nothing here is recorded or marked.
# Standard-library helpers for settings files, timing and folder paths.
import json
import os
import sys
import time
from glob import glob
from pathlib import Path
from typing import Literal

# Third-party pieces: HTTP client, Groq chat model, and LangChain base classes for models and embeddings.
import httpx
from dotenv import load_dotenv
from langchain_core.caches import InMemoryCache
from langchain_core.embeddings import Embeddings
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_groq import ChatGroq

# scikit-learn powers the offline TF-IDF embeddings; stop early with an install hint if it is missing.
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
except ImportError as e:
    raise ImportError(
        "scikit-learn is needed for TfidfEmbeddings. In a terminal run: "
        "pip install scikit-learn  (then restart the kernel)."
    ) from e

# Load a default .env file, if there is one, into the environment.
load_dotenv()


SOLUTIONS = {}  # step_id -> (builder function, one-sentence why, code text shown by show_solution)
CHECKPOINTS = {}  # part_id -> function(ns) that rebuilds the variables a late learner needs (canned/mock only)
CODES = {}  # part -> reference code texts, in order

# Look up per-million-token prices for the chosen GROQ_MODEL, used for cost estimates.
# MYR is preferred, then USD, then a built-in default.
with open("_groq_model_per_million_prices.json", mode="rb") as _model_prices_json:
    _MODEL_PRICES = json.load(_model_prices_json).get(
        os.environ.get("GROQ_MODEL", ""), {}
    )
    model_input_price = (_model_input_price := _MODEL_PRICES.get("input", {})).get(
        "MYR", _model_input_price.get("USD", 0.50)
    )
    model_output_price = (_model_output_price := _MODEL_PRICES.get("output", {})).get(
        "MYR", _model_output_price.get("USD", 2.00)
    )


def register(step_id, why, code=""):
    """Decorator used by the notebook: SOLUTIONS['d1-01-p2'] = the reference builder."""

    def deco(fn):
        SOLUTIONS[step_id] = (fn, why, code or getattr(fn, "__doc__", "") or "")
        return fn

    return deco


def checkpoint(part_id):
    """Decorator: register the function that rebuilds this Part's variables (see fast_forward)."""

    def deco(fn):
        CHECKPOINTS[part_id] = fn
        return fn

    return deco


def pick(step_id, attempt=None, is_using_solutions: bool = True):
    """Use your attempt, or the reference answer if USE_SOLUTIONS is True or your attempt is None."""
    if is_using_solutions or attempt is None:
        if step_id not in SOLUTIONS:
            raise KeyError(
                f"No reference answer registered for '{step_id}'. Click the heading 'Part A', "
                "choose Run > Run All Above Selected Cell, then run this cell again."
            )
        return SOLUTIONS[step_id][0]()
    return attempt


def check(step_id, ok, hint=""):
    """Plain self-check: tick or cross plus a hint. Unlimited re-runs. Nothing is stored or counted."""
    print(
        ("\u2714 " + step_id + ": looks right")
        if ok
        else (
            "\u2718 "
            + step_id
            + ": not yet. "
            + (hint or "Re-read the step, change your code, run again.")
        )
    )
    return bool(ok)


def show_solution(step_id):
    """Print the reference code and why it works. Keyboard-friendly: no copying needed, use pick()."""
    if step_id not in SOLUTIONS:
        print("No reference answer registered for", step_id)
        return
    _, why, code = SOLUTIONS[step_id]
    print(code.rstrip() or "(see the reference cell)")
    print("Why:", why)


def fast_forward(part_id):
    """Late or stuck? Rebuild this Part's variables from reference code (canned/mock, no API calls)."""

    if part_id not in CHECKPOINTS:
        print(
            "Not ready yet. Click the heading 'Part A', choose Run > Run All Above Selected Cell, "
            "then run this cell again. Known parts:",
            ", ".join(sorted(CHECKPOINTS)) or "(none registered yet)",
        )
        return

    if globals().get("LLM_MODE") == "groq":
        print("Note: fast-forward uses canned replies, not the live model.")

    CHECKPOINTS[part_id](globals())
    print("\u2714 Caught up to", part_id, "- continue with the next cell.")


# Load the workplace policy texts as system messages so every answer can be grounded in them.
def get_system_corpus(context: Literal["workplace"]) -> list[tuple[str, str]]:
    _corpus_dir: Path
    system_prompt: list[tuple[str, str]] = []

    if context == "workplace":
        _corpus_dir = Path.cwd() / "data" / "workplace*" / "*.txt"

    else:
        raise ValueError(f"Unsupported corpus: '{context}'")

    # Read each policy .txt file under data/workplace*/ as one system message.
    for _corpus_pathname in glob(str(_corpus_dir)):
        with open(_corpus_pathname, "r", encoding="utf-8") as _corpus_file:
            system_prompt.append(("system", _corpus_file.read()))

    # Finish with a rule that makes answers cite their source identifier.
    system_prompt.append(
        (
            "system",
            "Assistant should always cite source identifier to support claim when answering any asks.",
        )
    )

    return system_prompt


# Remember a code snippet under its Part so a late learner can rebuild that Part.
def state(part, code):
    CODES.setdefault(part, []).append(code.strip())


# Register a reference answer: keep its code text and a builder that runs it and returns result.
def ref(part, step, why, code):
    code = code.strip()
    if part:
        CODES.setdefault(part, []).append(code)

    # Run the reference code in a copy of the notebook variables and hand back its result.
    def build():
        ns = dict(globals())
        exec(code, ns)
        return ns["result"]

    register(step, why, code)(build)


# Define how to catch up: re-run the stored code of the listed Parts in order (used by fast_forward).
def rebuild(part, *parts):
    checkpoint(part)(lambda ns: [exec(c, ns) for p in parts for c in CODES[p]])


# Build the shared system messages once so later cells can reuse them.
SYSTEM_PROMPT = get_system_corpus("workplace")


# Guard the Python version: 3.14 or newer is too new for some course packages.
if sys.version_info[:2] >= (3, 14):
    raise RuntimeError(
        f"Python {sys.version.split()[0]} is too new for this course: some packages are not ready. "
        "Create the environment with Python 3.12 (3.11-3.13 work) and pick it as the notebook kernel."
    )

# Also stop on versions older than the supported 3.11.
assert (3, 11) <= sys.version_info[:2] <= (3, 13), (
    f"Use Python 3.11-3.13 (you have {sys.version.split()[0]})"
)
# Key lookup order:
#   1. file named in LM_ENV_FILE
#   2. ~/.langchain-mastery/.env (local, not synced)
#   3. .env next to notebooks.
# Only these four names are read from the file; everything else in it is ignored.

# Names read from the settings file, and folder names that suggest cloud-synced storage.
_ALLOWED = ("GROQ_API_KEY", "GROQ_MODEL", "LLM_MODE", "LM_ENV_FILE")
_SYNCED = ("onedrive", "sharepoint", "dropbox", "google drive", "googledrive", "icloud")


def _tilde(
    p,
):  # show the home folder as ~ so a shared screen does not show your user name
    p, home = str(p), str(Path.home())
    return (
        "~" + p[len(home) :]
        if p.lower().startswith(home.lower())
        else "(outside home) " + Path(p).name
    )


# Warn early if LM_ENV_FILE points to a file that does not exist.
_lm = os.getenv("LM_ENV_FILE", "").strip()
if _lm and not Path(_lm).is_file():
    print(
        "NOTE: LM_ENV_FILE is set but that file was not found, so it is ignored. Check the path."
    )

# Candidate settings files in lookup order; the first one that exists is used.
_env_files = [
    _lm,
    str(Path.home() / ".langchain-mastery" / ".env"),
    str(Path.cwd() / ".env"),
    str(Path.cwd() / "courseware" / ".env"),
]

# Copy only the allowed names from the chosen file into the environment, never overwriting values already set.
_env_file = next((p for p in _env_files if p and Path(p).is_file()), None)
if _env_file:
    try:
        from dotenv import dotenv_values

        for _k, _v in dotenv_values(_env_file).items():
            if (
                _k in _ALLOWED and _v is not None and _k not in os.environ
            ):  # existing variables win
                os.environ[_k] = _v

        # Tell the learner which file was used, with the home folder hidden.
        print(
            "Settings file:",
            _tilde(_env_file),
            "(only",
            ", ".join(_ALLOWED),
            "are read)",
        )

        # Warn if the file sits in a synced folder, where a key could be shared by accident.
        if any(w in _env_file.lower() for w in _SYNCED):
            print(
                "WARNING: your .env is in a synced folder (OneDrive, SharePoint, Dropbox, Google Drive "
                "or iCloud). Move it to ~/.langchain-mastery/.env"
            )

    except ImportError:
        print("python-dotenv not installed: using system environment variables only")
else:
    # With no settings file the notebook still runs in mock mode, so no key is needed.
    print("No .env file found: mock mode, no key needed.")

# Choose mock (offline, canned replies) or groq (live model); mock is the default.
LLM_MODE = os.getenv("LLM_MODE", "mock").strip().lower()
if LLM_MODE not in ("mock", "groq"):
    raise ValueError("LLM_MODE must be 'mock' or 'groq' (check your .env file)")

# Tracing switch: mock mode forces every tracing variable off; groq mode only warns if one is on.
_TRACING = ("LANGCHAIN_TRACING_V2", "LANGCHAIN_TRACING", "LANGSMITH_TRACING")
if LLM_MODE == "mock":
    for _t in _TRACING:
        os.environ[_t] = "false"

    print(
        "Mock mode: LangChain tracing is switched off for this session (nothing is sent anywhere)."
    )

elif any(os.getenv(_t, "").strip().lower() == "true" for _t in _TRACING):
    print(
        "WARNING: a LangChain tracing variable is on: live runs may be sent to a tracing service. Set it to false."
    )


class ScriptedToolChat(BaseChatModel):
    """Mock model for tool calling: replays a script. A str is a final answer; a dict
    {'name':..., 'args':{...}} (or a list of them) becomes AIMessage.tool_calls. Canned, not real."""

    # The script to replay and the position of the next step.
    script: list = []
    pos: int = 0

    @property
    def _llm_type(self):
        return "scripted-tool-chat"

    def bind_tools(self, tools, **kwargs):
        return self  # tools are ignored: the script decides

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        # Take the next scripted step, looping back to the start when the script runs out.
        step = (
            self.script[self.pos % len(self.script)] if self.script else "(mock reply)"
        )
        self.pos += 1

        # A text step becomes a final answer; a dict step becomes tool calls with made-up ids.
        if isinstance(step, str):
            msg = AIMessage(content=step)

        else:
            calls = step if isinstance(step, list) else [step]
            msg = AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": c["name"],
                        "args": c.get("args", {}),
                        "id": f"mock_{self.pos}_{i}",
                        "type": "tool_call",
                    }
                    for i, c in enumerate(calls)
                ],
            )

        return ChatResult(generations=[ChatGeneration(message=msg)])


def get_llm(responses=None, tool_calls=None):
    """Return a chat model. Groq in 'groq' mode; offline canned mock otherwise."""

    if LLM_MODE == "groq":
        # Live mode needs both a model name and a key; otherwise explain what is missing.
        model = os.getenv("GROQ_MODEL", "").strip()

        if not model or not os.getenv("GROQ_API_KEY", "").strip():
            raise RuntimeError(
                "Groq mode needs GROQ_API_KEY and GROQ_MODEL in .env "
                "(suggested model: openai/gpt-oss-20b; confirm in the Groq console), "
                "or set LLM_MODE=mock."
            )

        # Return the Groq model; temperature 0 and a cache keep replies repeatable.
        print("LIVE model:", model)
        return ChatGroq(
            cache=InMemoryCache(),
            model=model,
            temperature=0,
            http_client=httpx.Client(verify=False),
        )

    # Mock mode: an offline fake model, with scripted tool calls if given, otherwise canned text replies.
    print("MOCK: canned replies")
    if tool_calls:
        return ScriptedToolChat(script=list(tool_calls))

    return FakeListChatModel(responses=list(responses or ["(mock reply)"]))


class TraceLog:
    """Tiny local trace: one dict per step. This class sends nothing anywhere."""

    RATE_PER_1K_TOKENS = 0.0005  # PLACEHOLDER rate in RM per 1,000 tokens (not a real price); use your plan's

    def __init__(self):
        self.rows = []

    def add(self, step, tool="-", tokens_in=0, tokens_out=0, seconds=0.0):
        # Estimate cost from the total tokens at the placeholder rate (not a real price).
        cost = (tokens_in + tokens_out) / 1000 * self.RATE_PER_1K_TOKENS

        # Store one row per step so the trace can be shown as a table.
        self.rows.append(
            {
                "step": step,
                "tool": tool,
                "tokens_in": tokens_in,
                "tokens_out": tokens_out,
                "seconds": round(seconds, 3),
                "est_cost_myr": round(cost, 6),
            }
        )

    def timed(self, step, fn, *args, tool="-", **kw):
        # Run fn, time it, and log a step; token counts are rough guesses (about 4 characters per token).
        t0 = time.perf_counter()
        out = fn(*args, **kw)

        self.add(
            step,
            tool,
            len(str(args)) // 4,
            len(str(out)) // 4,
            time.perf_counter() - t0,
        )

        return out

    # Sum tokens, seconds and estimated cost over all recorded steps.
    def total(self):
        return {
            k: round(sum(r[k] for r in self.rows), 6)
            for k in ("tokens_in", "tokens_out", "seconds", "est_cost_myr")
        }


class TfidfEmbeddings(Embeddings):
    """Offline embeddings: scikit-learn TF-IDF fitted on your corpus, L2-normalised."""

    def __init__(self, corpus):
        # Fit the vocabulary on the corpus once so documents and questions share one vector space.
        self.vec = TfidfVectorizer(norm="l2", lowercase=True).fit(list(corpus))

    def embed_documents(self, texts):
        return self.vec.transform(list(texts)).toarray().tolist()

    def embed_query(self, text):
        return self.embed_documents([text])[0]


# ---- END HELPER ----
# ---- END CANONICAL SETUP ----

No .env file found: mock mode, no key needed.
Mock mode: LangChain tracing is switched off for this session (nothing is sent anywhere).


## Three rules

1. **Keep your key private.** Never paste it into a notebook, chat or email.
2. **Do not paste personal or company data.** Live mode sends text to a third party (Groq). Use only this course's fictional data.
3. **AI output needs a human check.** A neat answer can still be wrong. The more freedom an agent has, the more this matters.

Mock mode (the default) needs no key.

**Practices:** a blank starter looks like `my_attempt = None`. **If you leave it as `None`, the notebook quietly runs the reference answer instead, so Run All works even when you wrote nothing.** To learn, write your own and set `my_attempt` to it: then yours runs. `check(...)` prints ✔ or ✘ and a hint. `show_solution(step_id)` prints the reference code.

* `USE_SOLUTIONS`: `False` = you type the blanks,
* `USE_SOLUTIONS`: `True` = Run All reads the reference answers.

Reading first, then trying your own, teaches more.

In [4]:
# Notebook-specific imports and data path
from pathlib import Path
import json
from typing import Annotated, Literal

from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

try:
    from langgraph.graph import END, START, StateGraph
    from langgraph.graph.message import add_messages
    from langgraph.prebuilt import ToolNode, tools_condition

except ImportError as e:
    raise ImportError("Needs langgraph: pip install -r .requirements.txt") from e

# Folder with the course's fictional sample data files.
DATA = Path("data")
print("LLM_MODE =", LLM_MODE, "| data folder:", DATA)

LLM_MODE = mock | data folder: data


/usr/local/lib/python3.13/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [5]:
# Reference code for the Part A tools, kept as text so a late learner can rebuild them.
_state_d2_03_a_0 = r'''
RATES = {'Kuala Lumpur': 250, 'Penang': 250, 'Johor Bahru': 200}   # RM a night
class Stay(BaseModel):
    city: Literal['Kuala Lumpur', 'Penang', 'Johor Bahru']
    nights: int = Field(ge=1, le=30)
@tool(args_schema=Stay)
def hotel_cost(city: str, nights: int) -> str:
    """Hotel limit in RM for a stay in a Seri Maju city (policy SM-POL-08)."""
    return '%d nights, %s: RM%d max [SM-POL-08]' % (nights, city, RATES[city] * nights)
WIKI = json.loads((DATA / 'wiki_cache.json').read_text(encoding='utf-8'))
@tool
def wiki_lookup(topic: str) -> str:
    """Short background on a topic from the offline course cache (no internet)."""
    return WIKI.get(topic.strip().lower(), 'No cached summary for ' + repr(topic))
'''

_state_d2_03_a_1 = r'''
@tool
def leave_days(years: float) -> str:
    """Annual leave days by years of service (policy SM-POL-01)."""
    days = 14 if years < 2 else 16 if years <= 5 else 18
    return str(days) + ' days a year [SM-POL-01]'
class State(TypedDict):
    messages: Annotated[list, add_messages]   # new messages are appended
TOOLS = [hotel_cost, leave_days, wiki_lookup]
'''

_state_d2_03_a_2 = r"""
def make_chatbot(llm):
    bound = llm.bind_tools(TOOLS)   # tell the model which tools exist
    def chatbot(state: State):
        return {'messages': [bound.invoke(state['messages'])]}
    return chatbot
"""

# Store the Part A code texts, in order, so fast_forward can replay them.
state("d2-03-A", _state_d2_03_a_0)

state("d2-03-A", _state_d2_03_a_1)

# Reference answer for the route-prediction practice.
ref(
    None,
    "d2-03-predict",
    "Tool questions need the tools node.",
    "result = ['tools', 'direct', 'tools', 'tools']",
)

# Last Part A code piece: the chatbot node builder.
state("d2-03-A", _state_d2_03_a_2)

# Reference answer for the explain-a-route practice.
ref(
    None,
    "d2-03-explain",
    "Name the route, then say why.",
    "result = dict(route='chatbot -> tools -> chatbot',\n"
    "              explain='The model asked for a tool, so the router sent the run '\n"
    "                      'to tools; then the chatbot read the result and wrote the '\n"
    "                      'answer (canned).')",
)

# Reference answer for the graph-building practice.
ref(
    "d2-03-A",
    "d2-03-graph",
    "Tool calls go to tools; tools return to chatbot.",
    "def build_app(llm):\n"
    "    b = StateGraph(State)\n"
    "    b.add_node('chatbot', make_chatbot(llm)); b.add_node('tools', ToolNode(TOOLS))\n"
    "    b.add_edge(START, 'chatbot')\n"
    "    b.add_conditional_edges('chatbot', tools_condition)   # tools or END\n"
    "    b.add_edge('tools', 'chatbot')   # after a tool ran, the model reads the result\n"
    "    return b\n"
    "result = build_app",
)

# Part B constants as reference code: the tool belt, the fence and the hop budget.
state(
    "d2-03-B",
    "READ = {'hotel_cost', 'leave_days', 'wiki_lookup'}   # read-only tools on the belt\n"
    "RISKY = {'submit_claim', 'delete_file'}   # money, people, personal data: fenced\n"
    "MAX_HOPS = 4   # hop budget",
)

# Reference answer for the guard practice.
ref(
    "d2-03-B",
    "d2-03-guard",
    "Unknown tools: refuse. Risky tools: ask a human.",
    "def guard(call, dial=2):\n"
    "    name = call['name']\n"
    "    if name not in READ and name not in RISKY:\n"
    "        return 'refuse'\n"
    "    if name in RISKY:\n"
    "        return 'ask a human'\n"
    "    return 'ask a human' if dial == 1 else 'run'\n"
    "result = guard",
)

# Reference answer for the supervisor-or-relay practice.
ref(
    None,
    "d2-03-written",
    "Start with the supervisor: its route is easy to check.",
    "result = dict(choice='supervisor',\n"
    "              explain='A supervisor gives one clear route and a hop budget, so a '\n"
    "                      'person can follow each step. For money the dial stays at '\n"
    "                      'ask a human, because a wrong claim is costly.')",
)

In [6]:
import sys
import langchain
import langchain_core
import langchain_groq

print("Python:", sys.version)
print("LangChain:", getattr(langchain, "__version__", "unknown"))
print("LangChain Core:", getattr(langchain_core, "__version__", "unknown"))
print("LangChain Groq:", getattr(langchain_groq, "__version__", "unknown"))
print("LangChain file:", langchain.__file__)
print("Has verbose:", hasattr(langchain, "verbose"))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
LangChain: 0.3.30
LangChain Core: 0.3.86
LangChain Groq: 0.3.8
LangChain file: /usr/local/lib/python3.13/dist-packages/langchain/__init__.py
Has verbose: True


In [7]:
print("DATA:", "DATA" in globals())
print("get_llm:", "get_llm" in globals())
print("pick:", "pick" in globals())
print("check:", "check" in globals())
print("USE_SOLUTIONS:", "USE_SOLUTIONS" in globals())

DATA: True
get_llm: True
pick: True
check: True
USE_SOLUTIONS: True


## A Graph with a Router

### Idea: A Graph is a Rail Map (●)

- A **graph** has

  + **state** (here: the list of messages),
  + **nodes** (stations: functions that read the state and add to it) and
  + **edges** (tracks: which node runs next).
  
  See the picture at the top.

- A **conditional edge** is a junction: a small function decides where to go. The ready-made `tools_condition` says *tool calls in the last message? go to `tools`; otherwise finish*.
- The **tool loop** is two stations: `chatbot` (the model decides) and `tools` (runs the allow-listed tools). After `tools` the track goes **back to `chatbot`**, so the model can read the result.
- A **recursion limit** caps the number of steps, so a loop that never ends is stopped with an error instead of running for ever.
- **Mermaid text.** `get_graph().draw_mermaid()` gives the graph as plain text. Nothing is sent to an online service; paste it into any Mermaid viewer if you want a picture.
- **MOCK warning.** In mock mode a **canned** script chooses each route (labelled CANNED), so the table below proves the graph works, not that a real model routes well. The tools and the graph are real.

**Step 1 - The tools and the shared state (●)**

Run the next two cells as they are.

* `hotel_cost`, `wiki_lookup` and `leave_days` are read-only tools with typed arguments (as in notebook `d2-02-*.ipynb`).
* `TOOLS` is the allow-list.
* `make_chatbot` makes the node that asks the model.

In [10]:
# Fictional nightly hotel caps per city, so the demo never needs real prices.
RATES = {"Kuala Lumpur": 250, "Penang": 250, "Johor Bahru": 200}  # RM a night


class Stay(BaseModel):  # the typed schema: what the model may send
    city: Literal["Kuala Lumpur", "Penang", "Johor Bahru"]
    nights: int = Field(ge=1, le=30)


# Make this a tool, and reject any city or night count that does not fit the Stay schema.
@tool(args_schema=Stay)
def hotel_cost(city: str, nights: int) -> str:
    """Hotel limit in RM for a stay in a Seri Maju city (policy SM-POL-08)."""

    return "%d nights, %s: RM%d max [SM-POL-08]" % (nights, city, RATES[city] * nights)


# Load the offline wiki cache so lookups work without internet.
from pathlib import Path
import json

WIKI_PATH = Path("/content/wiki_cache.json")
WIKI = json.loads(WIKI_PATH.read_text(encoding="utf-8"))

print("Wiki topics loaded:", len(WIKI))
print("Available topics:", list(WIKI.keys()))

@tool
def wiki_lookup(topic: str) -> str:
    """Short background on a topic from the offline course cache (no internet)."""

    return WIKI.get(topic.strip().lower(), "No cached summary for " + repr(topic))


# Call the tool by hand once to see its output before a model ever uses it.
print(hotel_cost.invoke({"city": "Penang", "nights": 3}))

Wiki topics loaded: 7
Available topics: ['_note', 'fifa world cup', 'retrieval-augmented generation', 'large language model', 'react (agent pattern)', 'malaysia', 'kuala lumpur']
3 nights, Penang: RM750 max [SM-POL-08]


In [11]:
@tool
def leave_days(years: float) -> str:
    """Annual leave days by years of service (policy SM-POL-01)."""

    # Pick the tier by years of service: under 2, from 2 up to 5, then more than 5.
    days = 14 if years < 2 else 16 if years <= 5 else 18

    return str(days) + " days a year [SM-POL-01]"


class State(TypedDict):
    messages: Annotated[list, add_messages]  # new messages are appended, not replaced


TOOLS = [hotel_cost, leave_days, wiki_lookup]  # the allow-list


# Factory function, so each run can plug in its own model (canned or live).
def make_chatbot(llm):
    bound = llm.bind_tools(TOOLS)  # tell the model which tools exist

    # The graph node: ask the model about the conversation so far and add its reply to the state.
    def chatbot(state: State):
        return {"messages": [bound.invoke(state["messages"])]}

    return chatbot


# Confirm the allow-list holds exactly the three tools.
print([t.name for t in TOOLS])

['hotel_cost', 'leave_days', 'wiki_lookup']


**Step 2 and Practice 1 - Build the routing (●)**

Complete `build_app`. It already adds the two nodes and the first track. You add two things:

1. the conditional edge from `chatbot` that uses the ready-made `tools_condition`;
2. the edge from `tools` back to the node that reads a tool result.

Do **not** add a direct `chatbot -> END` edge: that would skip the tools.

**Leave `my_attempt = None` to use the reference.*

In [12]:
# Practice: finish the graph wiring. Two edges are missing, marked TODO below.
def my_build_app(llm):
    b = StateGraph(State)
    b.add_node("chatbot", make_chatbot(llm))
    b.add_node("tools", ToolNode(TOOLS))
    b.add_edge(START, "chatbot")

    # TODO 1: b.add_conditional_edges('chatbot', <the ready-made routing function>)

    # TODO 2: b.add_edge('tools', <the node that reads a tool result>)

    return b


my_attempt = None  # when ready, write: my_attempt = my_build_app

# Use your attempt, or the reference graph builder if my_attempt is still None.
build_app = pick("d2-03-graph", my_attempt, is_using_solutions=USE_SOLUTIONS)

# Build the graph with a placeholder model; only its wiring is inspected here.
B = build_app(ScriptedToolChat(script=["(unused)"]))

# Self-check the wiring: tools lead back to chatbot, and chatbot has no direct exit.
ok = ("tools", "chatbot") in B.edges and ("chatbot", END) not in B.edges
check(
    "d2-03-graph",
    ok,
    "After the chatbot, who decides what happens next, and "
    "what must happen after a tool runs? Then try show_solution().",
)

if not ok:
    build_app = pick("d2-03-graph", is_using_solutions=USE_SOLUTIONS)  # reference, so later cells run

✔ d2-03-graph: looks right


In [13]:
# Sample questions and the tool calls a canned model makes for them.
Q_HOTEL = "What is the hotel limit for 3 nights in Penang?"

HOTEL = {"name": "hotel_cost", "args": {"city": "Penang", "nights": 3}}
LEAVE = {"name": "leave_days", "args": {"years": 3}}
KL = {"name": "wiki_lookup", "args": {"topic": "Kuala Lumpur"}}

CASES = {  # question -> CANNED model script (used in mock mode only)
    Q_HOTEL: [HOTEL, "The limit is RM750 [SM-POL-08]."],
    "Hello! What can you do?": ["I can check hotel limits, leave days and topics."],
    "How many leave days after 3 years of service?": [LEAVE, "You get 16 days a year."],
    "Tell me about Kuala Lumpur.": [KL, "It is the capital of Malaysia."],
}

# Print the graph as Mermaid text so the routing arrows can be read.
print(build_app(ScriptedToolChat(script=["x"])).compile().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	chatbot(chatbot)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> chatbot;
	chatbot -.-> __end__;
	chatbot -.-> tools;
	tools --> chatbot;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



**Step 3 - Read the graph and run it (●)**

The text above is the graph in Mermaid. Read the arrows: the solid arrows always happen; the **dotted arrow** in this text is the conditional edge (a decision).

The picture at the top draws the tool detour as a dashed line: both mean "a decision".

Now `route_of` runs a question and lists which nodes ran, in order. Run the cell and read the four routes.

In [14]:
def model_for(script):  # mock: replay the CANNED script; live: the real model decides
    return ScriptedToolChat(script=list(script)) if LLM_MODE == "mock" else get_llm()


def route_of(script, question, limit=10):
    """The nodes that ran, in order, as 'chatbot -> tools -> chatbot'."""

    # Build a fresh graph for each question, using the model chosen above.
    app = build_app(model_for(script)).compile()

    # Start from the question; recursion_limit caps how many steps the run may take.
    inputs, cfg = {"messages": [("user", question)]}, {"recursion_limit": limit}

    # Stream per-node updates so the nodes can be listed in the order they ran.
    steps = app.stream(inputs, cfg, stream_mode="updates")

    return " -> ".join(next(iter(u)) for u in steps)


# Show each sample question with the route it took.
print("CANNED routes (mock)" if LLM_MODE == "mock" else "LIVE routes")
for q, script in CASES.items():
    print(route_of(script, q).ljust(30), "|", q)

CANNED routes (mock)
chatbot -> tools -> chatbot    | What is the hotel limit for 3 nights in Penang?
chatbot                        | Hello! What can you do?
chatbot -> tools -> chatbot    | How many leave days after 3 years of service?
chatbot -> tools -> chatbot    | Tell me about Kuala Lumpur.


**Practice 2 - Predict the route (●, no coding needed)**

For each of the four questions write `'tools'` (a tool is called first) or `'direct'` (the model answers alone).

Decide **before** you read the table. A DIFFERENT row is fine: it is information, and in live mode it can mean the model chose another route.

**Leave `my_pred = None` to see an example.*

In [15]:
# Practice: predict each route before looking at the table.
QS = list(CASES)
my_pred = None  # replace None with 4 words: 'tools' or 'direct'
PRED = pick("d2-03-predict", my_pred, is_using_solutions=USE_SOLUTIONS)
# Label a route 'tools' when the tools node ran, otherwise 'direct'.
kind = lambda route: "tools" if "tools" in route else "direct"
# Run every question to get the real routes to compare against.
ACTUAL = [kind(route_of(s, q)) for q, s in CASES.items()]
# Line up prediction and actual route, one question per row.
print("question".ljust(48), "| predicted | actual    | result")
for q, p, a in zip(QS, PRED, ACTUAL):
    flag = "MATCH" if p == a else "DIFFERENT"
    print(q[:48].ljust(48), "|", p.ljust(9), "|", a.ljust(9), "|", flag)
# Self-check: four predictions, each one of the two allowed words.
ok = len(PRED) == 4 and all(p in ("tools", "direct") for p in PRED)
check("d2-03-predict", ok, "Write four words, each tools or direct.")

question                                         | predicted | actual    | result
What is the hotel limit for 3 nights in Penang?  | tools     | tools     | MATCH
Hello! What can you do?                          | direct    | direct    | MATCH
How many leave days after 3 years of service?    | tools     | tools     | MATCH
Tell me about Kuala Lumpur.                      | tools     | tools     | MATCH
✔ d2-03-predict: looks right


True

### Check yourself

1. What are the three parts of a graph? <details><summary>Answer:</br></br></summary>State (the messages), nodes (functions that add to the state) and edges (which node runs next).</br></br></details>
2. What does the conditional edge decide? <details><summary>Answer:</br></br></summary>Whether the last message asked for tools (go to tools) or not (finish).</br></br></details>
3. Why does `tools` go back to `chatbot`? <details><summary>Answer:</br></br></summary>So the model can read the tool result and write the final answer.</br></br></details>
4. Does a canned route show that a real model routes well? <details><summary>Answer:</br></br></summary>No. A script chose the route. It shows the graph works; a live model may choose differently.</br></br></details>

You need `hotel_cost`, `leave_days`, `wiki_lookup` and `build_app`.

In [16]:
# Part B reuses the Part A names, so check which ones are missing.
need = ["hotel_cost", "leave_days", "wiki_lookup", "build_app"]
missing = [n for n in need if n not in globals()]

# Report readiness, and rebuild from reference code if a late learner is missing names.
print("All set for Part B." if not missing else "Missing: " + ", ".join(missing))

if missing and "d2-03-A" not in CODES:
    print("Click the heading Part A, Run > Run All Above Selected Cell, then rerun.")

elif missing:
    fast_forward("d2-03-A")  # rebuilds the names above from reference code

# Quick smoke test that the tools still answer.
print(leave_days.invoke({"years": 6}))
print(hotel_cost.invoke({"city": "Johor Bahru", "nights": 2}))

All set for Part B.
18 days a year [SM-POL-01]
2 nights, Johor Bahru: RM400 max [SM-POL-08]


## Budgets, Freedom and Supervisors

### Idea: The Autonomy Dial and Its Fence (●)

![A four-position dial runs from 1 ask first through 2 suggest and 3 act then tell to 4 act alone, with the needle at 2. On the right, a picket fence with a padlocked gate surrounds a yard where a robot may read policy, draft a reply or ask a human. Outside the fence are actions that need a human: send money and delete files. A bottom strip says to turn the dial down for money, people and personal data.](media/d2-03-autonomy-dial-fence.svg)

- **The dial** says how much the assistant may do without a person: 1 *ask first*, 2 *suggest* (a person sends), 3 *act, then tell*, 4 *act alone*. More freedom is faster and riskier.
- **The fence** is the **autonomy boundary**: what stays inside (read policy, draft a reply, ask a human) and what stays outside, always needing a person (**send money, delete files, change people's records, use personal data**).
- **Three guards** keep the assistant inside the fence:

  + the **allow-list** (only tools on the belt exist); (1)
  + a **budget** (hop, step or recursion limit); (2)
  + **ask a human first** for anything risky, at *every* dial setting. (3)

- **Rule of thumb for the Seri Maju helpdesk:** answers and look-ups at dial (2); anything that moves money (for example submitting a claim) always asks a person.

**A stop budget for graphs (●)**

This is guard (2) from the box above, in action. The canned model below is a **deliberate problem**: it asks for the same tool every time and never answers.

`recursion_limit` is the graph's step budget. When it is reached, LangGraph raises `GraphRecursionError`, and we catch it.

Run the cell: both limits stop the loop.

In [ ]:
# Needed to catch the error raised when the step budget runs out.
from langgraph.errors import GraphRecursionError

# A model that always asks for the hotel tool and never writes an answer: a deliberate endless loop.
Q_HOTEL = "What is the hotel limit for 3 nights in Penang?"
HOTEL = {"name": "hotel_cost", "args": {"city": "Penang", "nights": 3}}
LOOPING = ScriptedToolChat(script=[HOTEL])  # CANNED on purpose: never answers


# Run the looping graph under a given step budget and report how it ended.
def run_with_limit(limit):
    app = build_app(LOOPING).compile()

    try:
        app.invoke({"messages": [("user", Q_HOTEL)]}, {"recursion_limit": limit})

        return "finished"

    except GraphRecursionError:
        return "stopped by recursion_limit"


# Try a small and a larger budget: both should stop the loop.
for limit in (3, 6):
    print("limit", limit, "->", run_with_limit(limit))

# Self-check: the limit must turn the endless loop into a caught stop.
ok = run_with_limit(6) == "stopped by recursion_limit"
check("d2-03-limit", ok, "The endless loop must be stopped by the limit.")

**Practice 3 - Look back and explain a route in your own words (●, written)**

Back to the earlier graph: pick the hotel question. Name the route it took, then write two or three sentences:

+ why did the router send the run to `tools`?
+ what does a canned route prove (and not prove)?

**Leave `my_explain = None` to see an example, then write your own and compare.*

In [17]:
my_explain = None  # replace None with: dict(route='...', explain='...')

# Use your explanation, or the reference example if my_explain is still None.
ex = pick("d2-03-explain", my_explain, is_using_solutions=USE_SOLUTIONS)

# Self-check: the route mentions tools and the explanation is long enough to give a reason.
ok = "tools" in ex["route"] and len(ex["explain"].split()) >= 15
check("d2-03-explain", ok, "Name the route (with tools); write 15 or more words.")

print(ex["route"], "-", ex["explain"])

✔ d2-03-explain: looks right
chatbot -> tools -> chatbot - The model asked for a tool, so the router sent the run to tools; then the chatbot read the result and wrote the answer (canned).


In [18]:
READ = {"hotel_cost", "leave_days", "wiki_lookup"}  # read-only tools on the belt

RISKY = {"submit_claim", "delete_file"}  # money, people, personal data: fenced

# Test cases: tool name, autonomy dial setting and the answer a correct guard gives.
EXPECT = [
    ("hotel_cost", 2, "run"),
    ("hotel_cost", 1, "ask a human"),
    ("submit_claim", 4, "ask a human"),
    ("send_email", 3, "refuse"),
    ("delete_file", 2, "ask a human"),
]

# Print the table your guard must match.
print("call".ljust(13), "dial", "what a guard should say")

for name, dial, want in EXPECT:
    print(name.ljust(13), str(dial).ljust(4), want)

call          dial what a guard should say
hotel_cost    2    run
hotel_cost    1    ask a human
submit_claim  4    ask a human
send_email    3    refuse
delete_file   2    ask a human


**Practice 4 - Write the guard (●)**

Complete `guard(call, dial)`. It returns `'refuse'` for a tool that is not on the belt (not in `READ` or `RISKY`), `'ask a human'` for every risky tool at any dial, and for read-only tools `'ask a human'` at dial (1) and `'run'` otherwise.

The cell checks your function against the table above.

**Leave `my_attempt = None` to use the reference.*

In [19]:
# Practice: write the guard using the rules above.
def my_guard(call, dial=2):
    return None  # TODO: 'refuse', 'ask a human' or 'run' (see the rules above)


my_attempt = None  # when ready, write: my_attempt = my_guard

# Use your guard, or the reference guard if my_attempt is still None.
guard = pick("d2-03-guard", my_attempt, is_using_solutions=USE_SOLUTIONS)

# Run the guard on every row of the table.
got = [guard({"name": n, "args": {}}, d) for n, d, w in EXPECT]

# Self-check: the guard must give exactly the expected answers.
ok = got == [w for n, d, w in EXPECT]
check("d2-03-guard", ok, "Compare your answers with the table: " + str(got))

if not ok:
    guard = pick("d2-03-guard", is_using_solutions=USE_SOLUTIONS)  # reference, so later cells run

✔ d2-03-guard: looks right


### Idea: A Supervisor and Its Desks (●, read and run)

![Two panels. Left, supervisor: one manager with a clipboard in the middle sends tasks along solid lines to three workers and gets results back; caption clear route, easy to follow. Right, relay: three runners in lanes hand a striped baton along dashed arcs labelled hand-off; caption flexible but harder to trace. A bottom strip advises starting with the supervisor because it is easier to check.](media/d2-03-relay-baton.svg)

One agent with every tool gets crowded. **Two specialist desks** keep each job small: `hr_desk` (leave days) and `travel_desk` (hotel limits).

* A **supervisor** node reads the question, picks a desk, reads the desk's note, and decides again until it says `FINISH`.
* A **relay** has no manager: each desk hands the baton to the next one itself.

The relay is flexible but harder to trace, so **start with the supervisor**.

This is plain LangGraph: a few nodes and one conditional edge.

**Read and run the next four cells; nothing to fill in.** The supervisor's routing choices are **canned** (labelled CANNED) so the run is repeatable. A live model would make these choices itself.

Two safeguards are built in:

+ the router's reply is parsed safely (anything odd means `FINISH`), and
+ a **hop budget** (`MAX_HOPS`) ends the loop.

In [20]:
# operator.add below lets each desk append its note instead of replacing the list.
import operator


# Shared state for the team: the question, desk notes, the next desk and a hop counter.
class Team(TypedDict):
    question: str
    notes: Annotated[list, operator.add]  # each desk adds a note; lists are joined
    next: str
    hops: int


# The specialist desks the supervisor can pick from.
DESKS = ("hr_desk", "travel_desk")


def parse_route(text):
    """The first desk named in the reply; anything else is the safe fallback FINISH."""

    low = text.lower()

    return next((d for d in DESKS if d in low), "FINISH")


# Try a clear reply and an unclear one to see the safe FINISH fallback.
print(parse_route("Please ask travel_desk."), "|", parse_route("Sorry, not sure!"))

travel_desk | FINISH


In [21]:
MAX_HOPS = 4  # hop budget: the supervisor may decide at most this many times

# Count model calls so we can compare the cost of supervisor and relay later.
CALLS = {"model": 0}

# Instructions for the router model: pick exactly one desk or finish.
PROMPT = (
    "You are a supervisor. Desks: hr_desk (leave), travel_desk (hotels).\n"
    "Question: {question}\nAlready done: {done}\n"
    "Reply ONE word: hr_desk, travel_desk or FINISH."
)


# Build the supervisor node around a router model, so tests can swap in canned ones.
def make_supervisor(router):

    def supervisor(state: Team):
        # Stop deciding once the hop budget is used up.
        if state["hops"] >= MAX_HOPS:
            return {"next": "FINISH"}

        # Show the router what the desks have done so it does not repeat work.
        done = " | ".join(state["notes"]) or "nothing yet"

        CALLS["model"] += 1  # every model call is counted

        # Ask the router who goes next and record the hop.
        prompt = PROMPT.format(question=state["question"], done=done)
        reply = router.invoke(prompt).content

        return {"next": parse_route(reply), "hops": state["hops"] + 1}

    return supervisor

In [22]:
def hr_desk(state: Team):  # CANNED arguments: a real desk lets a model choose
    return {"notes": ["hr_desk: " + leave_days.invoke({"years": 3})]}


def travel_desk(state: Team):
    call = {"city": "Penang", "nights": 2}

    return {"notes": ["travel_desk: " + hotel_cost.invoke(call)]}


# Map each router word to a graph node; FINISH ends the run.
ROUTES = {"hr_desk": "hr_desk", "travel_desk": "travel_desk", "FINISH": END}


def build_supervisor(router):
    # Register the supervisor and the two desks as nodes.
    g = StateGraph(Team)
    g.add_node("supervisor", make_supervisor(router))
    g.add_node("hr_desk", hr_desk)
    g.add_node("travel_desk", travel_desk)

    # Wire the loop: start at the supervisor, follow its choice, and let each desk report back.
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", lambda s: s["next"], ROUTES)
    g.add_edge("hr_desk", "supervisor")
    g.add_edge("travel_desk", "supervisor")

    return g.compile()


# Confirm the desks are ready.
print("desks ready:", ", ".join(DESKS))

desks ready: hr_desk, travel_desk


In [23]:
# One question that needs both desks.
Q3 = "Leave days after 3 years, and the hotel limit for 2 nights in Penang?"


def run_supervisor(router_replies):
    # Reset the call counter so each run is measured on its own.
    CALLS["model"] = 0

    router = FakeListChatModel(responses=router_replies)  # CANNED router decisions

    # Run from an empty team state and keep the evidence we compare later.
    out = build_supervisor(router).invoke(
        {"question": Q3, "notes": [], "next": "", "hops": 0}
    )

    return {"notes": out["notes"], "hops": out["hops"], "model_calls": CALLS["model"]}


# Canned router: HR first, then travel, then FINISH.
R = run_supervisor(["hr_desk", "travel_desk", "FINISH"])

# Show what each desk found.
for note in R["notes"]:
    print(note)

# Compare hops with model calls: the supervisor decides once per hop.
print("supervisor hops:", R["hops"], "| model calls:", R["model_calls"])

# Draw a tiny supervisor graph as Mermaid text to see the loop shape.
demo = build_supervisor(FakeListChatModel(responses=["FINISH"]))
print(demo.get_graph().draw_mermaid())

hr_desk: 16 days a year [SM-POL-01]
travel_desk: 2 nights, Penang: RM500 max [SM-POL-08]
supervisor hops: 3 | model calls: 3
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	hr_desk(hr_desk)
	travel_desk(travel_desk)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	hr_desk --> supervisor;
	supervisor -. &nbsp;FINISH&nbsp; .-> __end__;
	supervisor -.-> hr_desk;
	supervisor -.-> travel_desk;
	travel_desk --> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



**The hop budget in action (●, run only)**

The next router is **canned and endless**: it never says `FINISH`.

Without a budget the graph would go round for ever. Read the output: the supervisor stops after exactly `MAX_HOPS` hops, so it made `MAX_HOPS` model calls and no more.

In [24]:
ENDLESS = ["hr_desk", "travel_desk"]  # CANNED router: never says FINISH

# Run the endless router: the hop budget has to cut it off.
R2 = run_supervisor(ENDLESS)

print("endless router stopped after", R2["hops"], "hops | calls:", R2["model_calls"])

# Self-check: it stopped exactly at the budget, with one model call per hop.
ok = R2["hops"] == MAX_HOPS and R2["model_calls"] == MAX_HOPS
check("d2-03-hops", ok, "The hop budget must stop an endless router at MAX_HOPS.")

endless router stopped after 4 hops | calls: 4
✔ d2-03-hops: looks right


True

**Practice 5 - Explain it in your own words (●, written)**

Choose `supervisor` or `relay` for the Seri Maju helpdesk, then write two or three sentences: why, and where you would set the autonomy dial for approving a claim.

**Leave `my_view = None` to see an example, then write your own and compare.*

In [25]:
my_view = None  # replace None with: dict(choice=<supervisor or relay>, explain='...')

# Use your view, or the reference example if my_view is still None.
v = pick("d2-03-written", my_view, is_using_solutions=USE_SOLUTIONS)

# Self-check: a valid choice and an explanation long enough to give reasons.
ok = v["choice"] in ("supervisor", "relay") and len(v["explain"].split()) >= 20
check("d2-03-written", ok, "Choose supervisor or relay and write 20 or more words.")

print(v["choice"], "-", v["explain"])

✔ d2-03-written: looks right
supervisor - A supervisor gives one clear route and a hop budget, so a person can follow each step. For money the dial stays at ask a human, because a wrong claim is costly.


### Check yourself

1. What does `recursion_limit` protect against? <details><summary>Answer:</br></br></summary>A loop that never ends: the run stops with an error at the step budget.</br></br></details>
2. What does the autonomy dial control? <details><summary>Answer:</br></br></summary>How much the assistant may do without a person: from ask first to act alone.</br></br></details>
3. Name the three guards. <details><summary>Answer:</br></br></summary>The allow-list (only belt tools exist), a budget (hops, steps, recursion limit) and ask a human first for risky actions.</br></br></details>
4. What stays outside the fence at every dial setting? <details><summary>Answer:</br></br></summary>Sending money, deleting files, changing people's records and using personal data: a human decides.</br></br></details>
5. How does a supervisor differ from a relay? <details><summary>Answer:</br></br></summary>A supervisor decides who works next and reads each note; in a relay each desk hands the baton on itself. The supervisor is easier to trace, so start there.</br></br></details>
6. Were the supervisor's choices a real model's? <details><summary>Answer:</br></br></summary>No: canned in mock mode. The graph, the tools, the guards and the hop budget are real.</br></br></details>

You should now have `guard`, `MAX_HOPS`, `run_supervisor` and `R`.

In [26]:
# Final readiness check: the guard and budgets from Part B should exist.
need = ["guard", "READ", "RISKY", "MAX_HOPS"]
missing = [n for n in need if n not in globals()]

print("d2-03 complete." if not missing else "Missing: " + ", ".join(missing))

if missing and not all(p in CODES for p in ("d2-03-A", "d2-03-B")):
    print("Click the heading Part A, Run > Run All Above Selected Cell, then rerun.")

elif missing:
    fast_forward("d2-03-B")  # rebuilds Part A and Part B names (canned, no API calls)

# Final smoke test: a risky call goes to a human even at the highest dial.
call = {"name": "submit_claim", "args": {}}
print("guard says:", guard(call, 4), "| hop budget:", MAX_HOPS)

d2-03 complete.
guard says: ask a human | hop budget: 4


## Recap and next step

- A **graph** is state, nodes and edges. A **conditional edge** routes tool calls to the tools node, and the tools node returns to the chatbot. `recursion_limit` stops endless loops. `draw_mermaid()` gives the graph as text.
- The **autonomy dial** sets how much the assistant does alone; the **fence** keeps money, people and personal data with a human. Three guards: allow-list, budget, ask a human first.
- A **supervisor** routes work to specialist desks and has a **hop budget**; a relay hands the baton on itself and is harder to trace. Start with the supervisor.
- Routes and router choices here are canned. Policies and amounts are fictional. AI output needs a human check.
- Next: **[d2-04-check-up-dashboard-and-wrap.ipynb](d2-04-check-up-dashboard-and-wrap.ipynb)** (check the assistant against a golden set, then close the two days).

### Stream a Run, Node by Node (◆)

`stream_mode='updates'` yields what each node added, as it happens.

Read the three lines: the chatbot asks for the tool, the tools node answers, the chatbot writes the final text. (Canned script.)

In [27]:
if "build_app" not in globals():
    fast_forward("d2-03-A")  # joined late? rebuild the Part A names (canned)

# A canned hotel request for the model to make.
call = {"name": "hotel_cost", "args": {"city": "Penang", "nights": 3}}

# Build the graph with a script: one tool call, then the final answer.
app = build_app(ScriptedToolChat(script=[call, "RM750."])).compile()  # CANNED

# Stream the run to see what each node adds, as it happens.
for update in app.stream(
    {"messages": [("user", "Hotel limit, 3 nights, Penang?")]}, stream_mode="updates"
):
    for node, out in update.items():
        # Show each node's text, or the tool it asks for when it has no text yet.
        last = out["messages"][-1]
        text = last.content or str([c["name"] for c in last.tool_calls])

        print("[" + node + "] " + last.type + ": " + text[:70])

[chatbot] ai: ['hotel_cost']
[tools] tool: 3 nights, Penang: RM750 max [SM-POL-08]
[chatbot] ai: RM750.


### The relay (swarm) and its ping-pong guard (◆)

In a relay there is no supervisor: after working, each desk decides who gets the baton next.

The next two cells build it and run it with **canned** hand-off choices. Then an endless hand-off (`PING`) shows the same hop budget stopping a ping-pong.

+ **E2 (a tool that calls a model):** if a tool itself calls a model, each tool call costs a hidden model call; count them.
+ **F1:** compare model calls for the supervisor and the relay on the same question.

In [29]:
# A relay desk does its own work, then decides the next hand-off itself (no supervisor).
def make_relay(name, desk, hand_llm):

    def node(state: Team):
        # Do this desk's own work first.
        note = desk(state)["notes"][0]

        CALLS["model"] += 1  # the hand-off decision is a model call

        # Ask the model whether to hand over, showing it all notes so far.
        seen = " | ".join(state["notes"] + [note])
        reply = hand_llm.invoke("Hand over? " + seen).content

        # Count the hop, finish when the budget is used up, otherwise follow the model's reply.
        hops = state["hops"] + 1
        nxt = "FINISH" if hops >= MAX_HOPS else parse_route(reply)

        # A desk that names itself ends the run instead of looping on itself.
        return {"notes": [note], "hops": hops, "next": "FINISH" if nxt == name else nxt}

    return node

In [28]:
def build_relay(hand_replies):
    # One canned hand-off model shared by both desks.
    hand, g = FakeListChatModel(responses=hand_replies), StateGraph(Team)  # CANNED

    # Wrap each desk as a relay node.
    g.add_node("hr_desk", make_relay("hr_desk", hr_desk, hand))
    g.add_node("travel_desk", make_relay("travel_desk", travel_desk, hand))
    # Always start the relay at hr_desk.
    g.add_conditional_edges(START, lambda s: "hr_desk", {"hr_desk": "hr_desk"})

    # After each desk, follow the desk's own 'next' choice.
    for d in DESKS:
        g.add_conditional_edges(d, lambda s: s["next"], ROUTES)

    return g.compile()

In [30]:
# The relay needs the supervisor cells from Part B to have run first.
if "run_supervisor" not in globals():
    print("Joined late? Run the Part B supervisor cells first, then rerun.")

else:

    # Run the relay from an empty team state and keep the evidence for comparison.
    def run_relay(hand_replies):
        CALLS["model"] = 0

        start = {"question": Q3, "notes": [], "next": "", "hops": 0}

        out = build_relay(hand_replies).invoke(start)

        return {
            "notes": out["notes"],
            "hops": out["hops"],
            "model_calls": CALLS["model"],
        }

    # Canned hand-offs: HR passes to travel, then DONE is not a desk, so the run finishes.
    relay = run_relay(["travel_desk", "DONE"])
    print(relay["notes"], "| runs:", relay["hops"], "| calls:", relay["model_calls"])

['hr_desk: 16 days a year [SM-POL-01]', 'travel_desk: 2 nights, Penang: RM500 max [SM-POL-08]'] | runs: 2 | calls: 2


In [31]:
# Only run this if the relay demo above ran.
if "relay" in globals():
    PING = ["travel_desk", "hr_desk"]  # CANNED and endless
    ping = run_relay(PING)

    # Show where the endless hand-offs were cut off.
    print("ping-pong stopped after", ping["hops"], "runs (MAX_HOPS =", MAX_HOPS, ")")

    # Self-check: the ping-pong must be cut off by the hop budget.
    check("d2-03-relay", ping["hops"] == MAX_HOPS, "The hop budget must stop it.")
    # Compare cost: model calls for the supervisor and for the relay.
    print("supervisor calls:", R["model_calls"], "| relay calls:", relay["model_calls"])

ping-pong stopped after 4 runs (MAX_HOPS = 4 )
✔ d2-03-relay: looks right
supervisor calls: 3 | relay calls: 2


### More graph ideas (■)

- Add a fourth tool (for example a palindrome checker, or a currency tool with a fixed, fictional RM rate) and add a case to `CASES`. Predict the route first.
- Try the *wrong* graph: add a direct `chatbot -> END` edge as well and compare the routes. What stops the tools from running?
- Add a `submit_claim` tool that only *writes a draft*, route it through `guard`, and show that the dial cannot make it run alone.

### A tool kit, step by step, and one agent against a supervisor (■)

Build the kit in four parts

+ tools,
+ the graph,
+ the guard,
+ the supervisor

in a fresh notebook, using only what you have seen here.

Then add another step: run the three questions (hotel, leave, both) through **one agent with both tools** and through the **supervisor**, and compare answers and model calls in a small table.

Say which you would trust, and why.

### A note on `create_react_agent` (■)

LangGraph has a ready-made shortcut, `create_react_agent`, that builds the same chatbot-and-tools graph in one line. It is handy once you understand the long form above. In newer LangGraph versions it may print a deprecation notice, so check the version you use.

In [32]:
import warnings

# The ready-made shortcut may be missing in some versions, so fall back gracefully.
try:
    from langgraph.prebuilt import create_react_agent

    # Canned script: one hotel tool call, then the final answer.
    script = [{"name": "hotel_cost", "args": {"city": "Penang", "nights": 3}}, "RM750."]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # the helper may warn that it is deprecated

        # One line builds the same kind of tool-loop graph that Part A wired by hand.
        SHORT = create_react_agent(ScriptedToolChat(script=script), TOOLS)  # CANNED

    # Run the shortcut graph and print its final answer.
    out = SHORT.invoke({"messages": [("user", "Hotel limit, 3 nights, Penang?")]})
    print("shortcut graph answer:", out["messages"][-1].content)

except ImportError:
    print("create_react_agent is not available in this version: use the manual graph.")

shortcut graph answer: RM750.
